<a href="https://colab.research.google.com/github/rychu1/ece147_final_project/blob/ryan/Colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [2]:
# install stuff
!git clone -b ryan https://github.com/rychu1/ece147_final_project.git
%cd /content/ece147_final_project
!pip install -e . --quiet

Cloning into 'ece147_final_project'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 161 (delta 42), reused 23 (delta 21), pack-reused 68 (from 3)
Receiving objects: 100% (161/161), 33.52 MiB | 11.43 MiB/s, done.
Resolving deltas: 100% (44/44), done.
Filtering content: 100% (18/18), 1.11 GiB | 15.97 MiB/s, done.
/content/ece147_final_project
  Preparing metadata (setup.py) ... done


### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [3]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.6/553.6 kB 13.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of typer to determine which version is

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

In [1]:
# get data to correct place to train/test

import os

!mkdir -p /content/ece147_final_project/data/
!unzip -q /content/drive/MyDrive/89335547.zip -d /content/ece147_final_project/data/

subfolder = "/content/ece147_final_project/data/89335547"
if os.path.exists(subfolder):
    !mv /content/ece147_final_project/data/89335547/* /content/ece147_final_project/data/
    !rm -rf /content/ece147_final_project/data/89335547

!ln -sf /content/ece147_final_project/data /content/data
!ls /content/ece147_final_project/data/


2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-04-1622861066-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-04-1622862148-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-05-1622884635-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5
2021-06-05-1622889105-keystrokes-dca-study@

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [2]:
%%javascript
function KeepAlive() {
    console.log("Keeping Colab alive...");
    document.querySelector("#top-toolbar > colab-connect-button")?.click();
    setTimeout(KeepAlive, 30000);
}
KeepAlive();

<IPython.core.display.Javascript object>

In [3]:
import threading, time, glob, os, shutil

os.makedirs("/content/drive/MyDrive/checkpoints", exist_ok=True)

def auto_save_to_drive():
    saved = set()
    while True:
        time.sleep(60)
        ckpts = [c for c in glob.glob("/content/logs/**/*.ckpt", recursive=True) if "last" not in c]
        for ckpt in ckpts:
            if ckpt not in saved:
                fname = os.path.basename(ckpt)
                shutil.copy(ckpt, f"/content/drive/MyDrive/checkpoints/{fname}")
                print(f"Auto-saved: {fname}")
                saved.add(ckpt)

threading.Thread(target=auto_save_to_drive, daemon=True).start()

# Single-user training

!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1
  # --multirun

[2026-03-10 04:13:40,316][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [4]:
import shutil, os

# Source — your hydra log folder from this run
LOG_DIR = "/content/logs/2026-03-10/04-13-40"

# Destination on Drive
DEST = "/content/drive/MyDrive/checkpoints"
os.makedirs(DEST, exist_ok=True)

# Copy the whole log folder to Drive
shutil.copytree(LOG_DIR, DEST, dirs_exist_ok=True)
print("Saved!")

Saved!


#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [5]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="/content/logs/2026-03-10/04-13-40/checkpoints/epoch=144-step=17400.ckpt" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

mismatched input '=' expecting <EOF>
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
